# Interactive Configuration GUI
### Real-time preview with nuclei-based filtering

**Load an image and adjust all parameters with instant visual feedback!**
**See which cells are excluded based on nuclei quality filtering.**

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches
import tifffile
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from cellpose import models, core
from scipy import ndimage
from skimage.morphology import skeletonize, remove_small_objects, disk, binary_closing
from skimage import measure
import pandas as pd
import json

# Import filtering modules
import sys
sys.path.append('.')
from filtering import NucleiFilter, CellFilter

print(' All imports successful')
print('\n' + '='*80)
print(' '*20 + 'INTERACTIVE CONFIGURATION GUI')
print('='*80)

In [ ]:
# Load existing config.json if it exists
config_path = Path('config.json')
if config_path.exists():
    with open(config_path, 'r') as f:
        existing_config = json.load(f)
    print('✓ Loaded existing config.json')
else:
    existing_config = None
    print(' No config.json found - will create new one on export')


In [ ]:
# Helper function to get config value with fallback
def get_config_value(path, default):
    """Get nested config value. path is like 'skeletonization.gaussian_sigma'"""
    if existing_config is None:
        return default
    
    keys = path.split('.')
    value = existing_config
    
    try:
        for key in keys:
            value = value[key]
        return value
    except (KeyError, TypeError):
        return default


In [ ]:
def sato_tubeness(image, sigmas):
    """Apply Sato tubeness filter for fiber detection"""
    responses = []
    
    for sigma in sigmas:
        Hxx = ndimage.gaussian_filter(image, sigma, order=[2, 0])
        Hxy = ndimage.gaussian_filter(image, sigma, order=[1, 1])
        Hyy = ndimage.gaussian_filter(image, sigma, order=[0, 2])
        
        tmp = np.sqrt((Hxx - Hyy)**2 + 4*Hxy**2)
        lambda1 = 0.5 * (Hxx + Hyy + tmp)
        lambda2 = 0.5 * (Hxx + Hyy - tmp)
        
        tubeness = np.zeros_like(image, dtype=np.float32)
        mask = lambda2 < 0
        tubeness[mask] = np.abs(lambda2[mask])
        responses.append(tubeness * sigma**2)
    
    return np.max(responses, axis=0)

print('Sato tubeness filter loaded')

## Step 1: Load Test Image

In [ ]:
# Set your input directory
input_dir = Path(r'C:\Work\test')  # Change this!

# Find all images
image_files = list(input_dir.glob('*.tif*'))
if len(image_files) == 0:
    print('No images found!')
    print(f'   Looking in: {input_dir}')
else:
    print(f'Found {len(image_files)} images')
    print(f'   Loading: {image_files[0].name}')
    
    # Load first image
    img_stack = tifffile.imread(image_files[0])
    print(f'   Image shape: {img_stack.shape}')
    
    # Handle 4D [Z, C, Y, X]
    if img_stack.ndim == 4:
        print(f'   4D stack detected - applying max projection')
        actin_raw = np.max(img_stack[:, 0, :, :], axis=0)
        nuclei_raw = np.max(img_stack[:, 1, :, :], axis=0)
    elif img_stack.ndim == 3 and img_stack.shape[0] == 2:
        actin_raw = img_stack[0]
        nuclei_raw = img_stack[1]
    else:
        print('Unexpected image format')
    
    print('Image loaded successfully')

## Step 2: Brightness/Contrast Adjustment

In [ ]:
style = {'description_width': '180px'}
layout_medium = widgets.Layout(width='400px')

display(HTML('<h3 style="color: #2E86AB;">ðŸ”† Brightness & Contrast</h3>'))
display(HTML('<h4>Actin/Cytoskeleton Channel:</h4>'))

bright_actin = widgets.FloatSlider(
    value=1.9, min=0.1, max=3.0, step=0.05,
    description='Brightness:', readout_format='.2f',
    style=style, layout=layout_medium, continuous_update=True
)
contrast_actin = widgets.FloatSlider(
    value=1.75, min=0.1, max=3.0, step=0.05,
    description='Contrast:', readout_format='.2f',
    style=style, layout=layout_medium, continuous_update=True
)
gamma_actin = widgets.FloatSlider(
    value=1.0, min=0.1, max=3.0, step=0.05,
    description='Gamma:', readout_format='.2f',
    style=style, layout=layout_medium, continuous_update=True
)

display(bright_actin, contrast_actin, gamma_actin)

display(HTML('<h4>Nuclei Channel:</h4>'))

bright_nuclei = widgets.FloatSlider(
    value=1.55, min=0.1, max=3.0, step=0.05,
    description='Brightness:', readout_format='.2f',
    style=style, layout=layout_medium, continuous_update=True
)
contrast_nuclei = widgets.FloatSlider(
    value=1.5, min=0.1, max=3.0, step=0.05,
    description='Contrast:', readout_format='.2f',
    style=style, layout=layout_medium, continuous_update=True
)
gamma_nuclei = widgets.FloatSlider(
    value=1.0, min=0.1, max=3.0, step=0.05,
    description='Gamma:', readout_format='.2f',
    style=style, layout=layout_medium, continuous_update=True
)

display(bright_nuclei, contrast_nuclei, gamma_nuclei)

brightness_output = widgets.Output()

def adjust_image(img, brightness, contrast, gamma):
    img = img.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-10)
    img = img * brightness
    mean = img.mean()
    img = (img - mean) * contrast + mean
    img = np.power(img, 1.0 / gamma)
    img = np.clip(img, 0, 1)
    return img

def update_brightness_preview(*args):
    with brightness_output:
        clear_output(wait=True)
        
        actin_adj = adjust_image(actin_raw, bright_actin.value, contrast_actin.value, gamma_actin.value)
        nuclei_adj = adjust_image(nuclei_raw, bright_nuclei.value, contrast_nuclei.value, gamma_nuclei.value)
        
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        
        axes[0, 0].imshow(actin_raw, cmap='gray')
        axes[0, 0].set_title('Actin - Original', fontsize=14, fontweight='bold')
        axes[0, 0].axis('off')
        
        axes[0, 1].imshow(actin_adj, cmap='gray')
        axes[0, 1].set_title('Actin - Adjusted', fontsize=14, fontweight='bold', color='green')
        axes[0, 1].axis('off')
        
        axes[1, 0].imshow(nuclei_raw, cmap='gray')
        axes[1, 0].set_title('Nuclei - Original', fontsize=14, fontweight='bold')
        axes[1, 0].axis('off')
        
        axes[1, 1].imshow(nuclei_adj, cmap='gray')
        axes[1, 1].set_title('Nuclei - Adjusted', fontsize=14, fontweight='bold', color='green')
        axes[1, 1].axis('off')
        
        plt.tight_layout()
        plt.show()

# Link widgets
for widget in [bright_actin, contrast_actin, gamma_actin, bright_nuclei, contrast_nuclei, gamma_nuclei]:
    widget.observe(update_brightness_preview, 'value')

display(brightness_output)
update_brightness_preview()

## Step 3: Segmentation Parameters

In [ ]:
display(HTML('<h3 style="color: #2E86AB;">Segmentation Parameters</h3>'))
display(HTML('<h4>Cell Segmentation:</h4>'))

cell_diameter = widgets.IntSlider(
    value=0, min=0, max=200, step=5,
    description='Diameter (0=auto):', style=style, layout=layout_medium, continuous_update=False
)
cell_flow_thresh = widgets.FloatSlider(
    value=0.25, min=0.0, max=1.0, step=0.05,
    description='Flow Threshold:', readout_format='.2f',
    style=style, layout=layout_medium, continuous_update=False
)
cell_prob_thresh = widgets.FloatSlider(
    value=-1.0, min=-6.0, max=6.0, step=0.5,
    description='Prob Threshold:', readout_format='.1f',
    style=style, layout=layout_medium, continuous_update=False
)

display(cell_diameter, cell_flow_thresh, cell_prob_thresh)

display(HTML('<h4>Nuclei Segmentation:</h4>'))

nucleus_diameter = widgets.IntSlider(
    value=0, min=0, max=200, step=5,
    description='Diameter (0=auto):', style=style, layout=layout_medium, continuous_update=False
)
nucleus_flow_thresh = widgets.FloatSlider(
    value=0.4, min=0.0, max=1.0, step=0.05,
    description='Flow Threshold:', readout_format='.2f',
    style=style, layout=layout_medium, continuous_update=False
)
nucleus_prob_thresh = widgets.FloatSlider(
    value=0.0, min=-6.0, max=6.0, step=0.5,
    description='Prob Threshold:', readout_format='.1f',
    style=style, layout=layout_medium, continuous_update=False
)

display(nucleus_diameter, nucleus_flow_thresh, nucleus_prob_thresh)

segment_button = widgets.Button(description=' Run Segmentation', button_style='success')
segment_output = widgets.Output()

# Store masks globally
latest_cell_masks = None
latest_valid_cells = None
latest_nucleus_masks = None

def run_segmentation(b):
    global latest_cell_masks, latest_nucleus_masks
    
    with segment_output:
        clear_output(wait=True)
        print('Running segmentation...')
        
        # Get adjusted images
        actin_adj = adjust_image(actin_raw, bright_actin.value, contrast_actin.value, gamma_actin.value)
        nuclei_adj = adjust_image(nuclei_raw, bright_nuclei.value, contrast_nuclei.value, gamma_nuclei.value)
        
        # Convert to uint16
        actin_adj = (actin_adj * 65535).astype(np.uint16)
        nuclei_adj = (nuclei_adj * 65535).astype(np.uint16)
        
        # Initialize models - Cellpose v4 API
        use_gpu = core.use_gpu()
        print(f'   GPU available: {use_gpu}')
        
        # Cellpose v4 - CellposeModel with cpsam pretrained model
        cell_model = models.CellposeModel(gpu=use_gpu, pretrained_model='cpsam')
        nucleus_model = models.CellposeModel(gpu=use_gpu, pretrained_model='cpsam')
        
        # Segment cells - stack actin and nuclei
        print('   Segmenting cells...')
        stacked_image = np.stack([actin_adj, nuclei_adj], axis=-1)
        
        # Cellpose v4 returns 3 values: masks, flows, styles
        latest_cell_masks, _, _ = cell_model.eval(
            stacked_image,
            diameter=cell_diameter.value if cell_diameter.value > 0 else None,
            flow_threshold=cell_flow_thresh.value,
            cellprob_threshold=cell_prob_thresh.value
        )
        
        # Segment nuclei
        print('   Segmenting nuclei...')
        latest_nucleus_masks, _, _ = nucleus_model.eval(
            nuclei_adj,
            diameter=nucleus_diameter.value if nucleus_diameter.value > 0 else None,
            flow_threshold=nucleus_flow_thresh.value,
            cellprob_threshold=nucleus_prob_thresh.value
        )
        
        # Display results
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        
        axes[0, 0].imshow(actin_adj, cmap='gray')
        axes[0, 0].set_title('Actin Image', fontsize=14, fontweight='bold')
        axes[0, 0].axis('off')
        
        axes[0, 1].imshow(latest_cell_masks, cmap='tab20')
        axes[0, 1].set_title(f'Cell Masks (n={latest_cell_masks.max()})', fontsize=14, fontweight='bold', color='green')
        axes[0, 1].axis('off')
        
        axes[1, 0].imshow(nuclei_adj, cmap='gray')
        axes[1, 0].set_title('Nuclei Image', fontsize=14, fontweight='bold')
        axes[1, 0].axis('off')
        
        axes[1, 1].imshow(latest_nucleus_masks, cmap='tab20')
        axes[1, 1].set_title(f'Nucleus Masks (n={latest_nucleus_masks.max()})', fontsize=14, fontweight='bold', color='green')
        axes[1, 1].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        print('\nSegmentation complete!')
        print(f'   Found {latest_cell_masks.max()} cells')
        print(f'   Found {latest_nucleus_masks.max()} nuclei')
        print('\nNow proceed to Step 4 to configure filtering')

segment_button.on_click(run_segmentation)

display(segment_button)
display(segment_output)

## Step 4: Nuclei Quality Filtering (Interactive Preview)

In [ ]:
display(HTML('<h3 style="color: #2E86AB;">🧬 Quality Filtering Parameters</h3>'))
display(HTML('<p><i>Adjust parameters and see which cells pass/fail filtering in real-time</i></p>'))

# =============================================================================
# NUCLEUS FILTERING
# =============================================================================
display(HTML('<hr style="border: 2px solid #2E86AB; margin: 20px 0;">'))
display(HTML('<h3 style="color: #E63946; background-color: #FFE5E5; padding: 10px; border-radius: 5px;">🔴 NUCLEUS FILTERING</h3>'))
display(HTML('<p><i>Filter out dividing, irregular, or bacteria-infected nuclei</i></p>'))

display(HTML('<h4>📏 Nucleus Size:</h4>'))
min_nucleus_area = widgets.IntText(
    value=get_config_value('nucleus_filtering.min_area', 100), 
    description='Min Area (px²):', 
    style=style
)
max_nucleus_area = widgets.IntText(
    value=get_config_value('nucleus_filtering.max_area', 800), 
    description='Max Area (px²):', 
    style=style
)
display(HTML('<small>Tip: max_area=800 rejects double/fused nuclei</small>'))
display(min_nucleus_area, max_nucleus_area)

display(HTML('<h4>🔷 Nucleus Shape:</h4>'))
min_nucleus_solidity = widgets.FloatSlider(
    value=get_config_value('nucleus_filtering.min_solidity', 0.85), 
    min=0.0, max=1.0, step=0.01,
    description='Min Solidity:', readout_format='.2f',
    style=style, layout=layout_medium
)
display(HTML('<small>Higher = more regular shape (rejects irregular/damaged nuclei)</small>'))
display(min_nucleus_solidity)

min_nucleus_eccentricity = widgets.FloatSlider(
    value=get_config_value('nucleus_filtering.min_eccentricity', 0.0), 
    min=0.0, max=1.0, step=0.01,
    description='Min Eccentricity:', readout_format='.2f',
    style=style, layout=layout_medium
)
max_nucleus_eccentricity = widgets.FloatSlider(
    value=get_config_value('nucleus_filtering.max_eccentricity', 0.75), 
    min=0.0, max=1.0, step=0.01,
    description='Max Eccentricity:', readout_format='.2f',
    style=style, layout=layout_medium
)
display(HTML('<small>⚠️ IMPORTANT: Lower max (e.g., 0.75) rejects elongated/dividing nuclei</small>'))
display(min_nucleus_eccentricity, max_nucleus_eccentricity)

min_nucleus_roundness = widgets.FloatSlider(
    value=get_config_value('nucleus_filtering.min_roundness', 0.6), 
    min=0.0, max=1.0, step=0.01,
    description='Min Roundness:', readout_format='.2f',
    style=style, layout=layout_medium
)
display(HTML('<small>⚠️ IMPORTANT: Higher min (e.g., 0.6) rejects dumbbell-shaped dividing nuclei</small>'))
display(min_nucleus_roundness)

display(HTML('<h4>💡 Nucleus Intensity:</h4>'))
min_nucleus_intensity = widgets.FloatSlider(
    value=get_config_value('nucleus_filtering.min_mean_intensity', 0.05), 
    min=0.0, max=1.0, step=0.01,
    description='Min Mean Intensity:', readout_format='.2f',
    style=style, layout=layout_medium
)
max_intensity_std = widgets.FloatSlider(
    value=get_config_value('nucleus_filtering.max_intensity_std', 2.0), 
    min=0.0, max=3.0, step=0.1,
    description='Max Intensity Std:', readout_format='.2f',
    style=style, layout=layout_medium
)
display(HTML('<small>Detects bacteria-infected or damaged nuclei (high variance)</small>'))
display(min_nucleus_intensity, max_intensity_std)

# =============================================================================
# CELL FILTERING
# =============================================================================
display(HTML('<hr style="border: 2px solid #2E86AB; margin: 20px 0;">'))
display(HTML('<h3 style="color: #1D3557; background-color: #D4E6F1; padding: 10px; border-radius: 5px;">🔵 CELL FILTERING</h3>'))
display(HTML('<p><i>Filter cells by size, shape, and nucleus association</i></p>'))

display(HTML('<h4>📏 Cell Size:</h4>'))
min_cell_area = widgets.IntText(
    value=get_config_value('cell_filtering.min_area', 500), 
    description='Min Area (px²):', 
    style=style
)
max_cell_area = widgets.IntText(
    value=get_config_value('cell_filtering.max_area', 5000), 
    description='Max Area (px²):', 
    style=style
)
display(HTML('<small>Tip: Tight range (500-5000) gives more uniform cell population</small>'))
display(min_cell_area, max_cell_area)

display(HTML('<h4>🔷 Cell Shape:</h4>'))
min_cell_solidity = widgets.FloatSlider(
    value=get_config_value('cell_filtering.min_solidity', 0.75), 
    min=0.0, max=1.0, step=0.01,
    description='Min Solidity:', readout_format='.2f',
    style=style, layout=layout_medium
)
display(HTML('<small>Higher = more regular shape (rejects very irregular cells)</small>'))
display(min_cell_solidity)

max_cell_eccentricity = widgets.FloatSlider(
    value=get_config_value('cell_filtering.max_eccentricity', 0.85), 
    min=0.0, max=1.0, step=0.01,
    description='Max Eccentricity:', readout_format='.2f',
    style=style, layout=layout_medium
)
display(HTML('<small>⚠️ NEW: Lower max (e.g., 0.85) rejects elongated/dividing cells</small>'))
display(max_cell_eccentricity)

display(HTML('<h4>🎯 Cell-Nucleus Association:</h4>'))
reject_multiple_nuclei = widgets.Checkbox(
    value=get_config_value('cell_filtering.reject_multiple_nuclei', True),
    description='Reject cells with 2+ nuclei',
    style={'description_width': 'initial'}
)
display(HTML('<small>⚠️ NEW: Enable to exclude dividing cells and cell clusters</small>'))
display(reject_multiple_nuclei)

border_buffer = widgets.IntSlider(
    value=get_config_value('cell_filtering.border_buffer', 10), 
    min=0, max=50, step=1,
    description='Border Buffer (px):', style=style, layout=layout_medium
)
display(HTML('<small>Excludes cells touching image edges</small>'))
display(border_buffer)

display(HTML('<hr style="border: 2px solid #2E86AB; margin: 20px 0;">'))

filter_button = widgets.Button(description='🧬 Apply Filtering', button_style='primary')
filter_output = widgets.Output()

def run_filtering(b):
    with filter_output:
        clear_output(wait=True)
        
        if latest_cell_masks is None or latest_nucleus_masks is None:
            print('âŒ Please run segmentation first!')
            return
        
        print('ðŸ§¬ Applying nuclei quality filtering...')
        
        # Create config
        config = {
            'nucleus_filtering': {
                'min_area': min_nucleus_area.value,
                'max_area': max_nucleus_area.value,
                'min_solidity': min_nucleus_solidity.value,
                'min_eccentricity': min_nucleus_eccentricity.value,
                'max_eccentricity': max_nucleus_eccentricity.value,
                'min_roundness': min_nucleus_roundness.value,
                'min_mean_intensity': min_nucleus_intensity.value,
                'max_intensity_std': max_intensity_std.value
            },
            'cell_filtering': {
                'min_area': min_cell_area.value,
                'max_area': max_cell_area.value,
                'min_solidity': min_cell_solidity.value,
                'max_eccentricity': max_cell_eccentricity.value,
                'reject_multiple_nuclei': reject_multiple_nuclei.value,
                'max_eccentricity': max_cell_eccentricity.value,
                'reject_multiple_nuclei': reject_multiple_nuclei.value,
                'border_buffer': border_buffer.value,
                'require_valid_nucleus': True
            }
        }
        
        # Get adjusted nuclei image
        nuclei_adj = adjust_image(nuclei_raw, bright_nuclei.value, contrast_nuclei.value, gamma_nuclei.value)
        nuclei_adj = (nuclei_adj * 65535).astype(np.uint16)
        
        # Filter nuclei
        nuclei_filter = NucleiFilter(config)
        nuclei_result = nuclei_filter.filter_nuclei(latest_nucleus_masks, nuclei_adj, 'test_image')
        
        # Filter cells
        cell_filter = CellFilter(config)
        cell_result = cell_filter.filter_cells(
            latest_cell_masks,
            latest_nucleus_masks,
            nuclei_result['valid_nuclei'],
            'test_image'
        )
        
        # Create visualization
        fig = plt.figure(figsize=(18, 12))
        gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
        
        # Top row: Original data
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.imshow(nuclei_adj, cmap='gray')
        ax1.set_title('Nuclei Image', fontsize=12, fontweight='bold')
        ax1.axis('off')
        
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.imshow(latest_nucleus_masks, cmap='tab20')
        ax2.set_title(f'All Nuclei (n={latest_nucleus_masks.max()})', fontsize=12, fontweight='bold')
        ax2.axis('off')
        
        ax3 = fig.add_subplot(gs[0, 2])
        ax3.imshow(latest_cell_masks, cmap='tab20')
        ax3.set_title(f'All Cells (n={latest_cell_masks.max()})', fontsize=12, fontweight='bold')
        ax3.axis('off')
        
        # Middle row: Filtered nuclei
        ax4 = fig.add_subplot(gs[1, 0])
        valid_nuc_mask = np.isin(latest_nucleus_masks, nuclei_result['valid_nuclei'])
        ax4.imshow(nuclei_adj, cmap='gray', alpha=0.5)
        ax4.imshow(valid_nuc_mask, cmap='Greens', alpha=0.5)
        ax4.set_title(f'Valid Nuclei (n={len(nuclei_result["valid_nuclei"])})', 
                      fontsize=12, fontweight='bold', color='green')
        ax4.axis('off')
        
        ax5 = fig.add_subplot(gs[1, 1])
        excluded_nuc_ids = [e['nucleus_id'] for e in nuclei_result['excluded_nuclei']]
        excluded_nuc_mask = np.isin(latest_nucleus_masks, excluded_nuc_ids)
        ax5.imshow(nuclei_adj, cmap='gray', alpha=0.5)
        ax5.imshow(excluded_nuc_mask, cmap='Reds', alpha=0.5)
        ax5.set_title(f'Excluded Nuclei (n={len(excluded_nuc_ids)})', 
                      fontsize=12, fontweight='bold', color='red')
        ax5.axis('off')
        
        ax6 = fig.add_subplot(gs[1, 2])
        # Show nuclei color-coded by exclusion reason
        reason_map = np.zeros_like(latest_nucleus_masks, dtype=float)
        for excluded in nuclei_result['excluded_nuclei']:
            mask = latest_nucleus_masks == excluded['nucleus_id']
            if excluded['reason'] == 'too_small':
                reason_map[mask] = 1
            elif excluded['reason'] == 'too_large':
                reason_map[mask] = 2
            elif excluded['reason'] == 'irregular_shape':
                reason_map[mask] = 3
            elif excluded['reason'] in ['too_round', 'too_elongated', 'irregular_boundary']:
                reason_map[mask] = 4
            elif excluded['reason'] in ['low_intensity', 'high_variance_bacteria']:
                reason_map[mask] = 5
        ax6.imshow(nuclei_adj, cmap='gray', alpha=0.3)
        im = ax6.imshow(reason_map, cmap='jet', alpha=0.6, vmin=0, vmax=5)
        ax6.set_title('Exclusion Reasons', fontsize=12, fontweight='bold')
        ax6.axis('off')
        
        # Bottom row: Cell filtering results
        ax7 = fig.add_subplot(gs[2, 0])
        valid_cell_ids = [c['cell_id'] for c in cell_result['valid_cells']]
        valid_cell_mask = np.isin(latest_cell_masks, valid_cell_ids)
        actin_adj = adjust_image(actin_raw, bright_actin.value, contrast_actin.value, gamma_actin.value)
        ax7.imshow(actin_adj, cmap='gray', alpha=0.5)
        ax7.imshow(valid_cell_mask, cmap='Greens', alpha=0.5)
        ax7.set_title(f'Valid Cells (n={len(valid_cell_ids)})', 
                      fontsize=12, fontweight='bold', color='green')
        ax7.axis('off')
        
        ax8 = fig.add_subplot(gs[2, 1])
        excluded_cell_ids = [c['cell_id'] for c in cell_result['excluded_cells']]
        excluded_cell_mask = np.isin(latest_cell_masks, excluded_cell_ids)
        ax8.imshow(actin_adj, cmap='gray', alpha=0.5)
        ax8.imshow(excluded_cell_mask, cmap='Reds', alpha=0.5)
        ax8.set_title(f'Excluded Cells (n={len(excluded_cell_ids)})', 
                      fontsize=12, fontweight='bold', color='red')
        ax8.axis('off')
        
        ax9 = fig.add_subplot(gs[2, 2])
        # Show exclusion reasons for cells
        cell_reason_map = np.zeros_like(latest_cell_masks, dtype=float)
        for excluded in cell_result['excluded_cells']:
            mask = latest_cell_masks == excluded['cell_id']
            if excluded['reason'] == 'too_small':
                cell_reason_map[mask] = 1
            elif excluded['reason'] == 'too_large':
                cell_reason_map[mask] = 2
            elif excluded['reason'] == 'irregular_shape':
                cell_reason_map[mask] = 3
            elif excluded['reason'] == 'on_border':
                cell_reason_map[mask] = 4
            elif excluded['reason'] == 'no_valid_nucleus':
                cell_reason_map[mask] = 5
        ax9.imshow(actin_adj, cmap='gray', alpha=0.3)
        im2 = ax9.imshow(cell_reason_map, cmap='jet', alpha=0.6, vmin=0, vmax=5)
        ax9.set_title('Cell Exclusion Reasons', fontsize=12, fontweight='bold')
        ax9.axis('off')
        
        plt.suptitle('Nuclei-Based Filtering Results', fontsize=16, fontweight='bold', y=0.98)
        plt.show()
        
        # Print statistics
        print('\n' + '='*80)
        print('FILTERING STATISTICS')
        print('='*80)
        print(f'\nNUCLEI:')
        print(f'   Total detected: {latest_nucleus_masks.max()}')
        print(f'   âœ… Valid: {len(nuclei_result["valid_nuclei"])} ({100*len(nuclei_result["valid_nuclei"])/latest_nucleus_masks.max():.1f}%)')
        print(f'   âŒ Excluded: {len(nuclei_result["excluded_nuclei"])} ({100*len(nuclei_result["excluded_nuclei"])/latest_nucleus_masks.max():.1f}%)')
        
        if nuclei_result['excluded_nuclei']:
            print(f'\n   Exclusion reasons:')
            from collections import Counter
            reasons = Counter([e['reason'] for e in nuclei_result['excluded_nuclei']])
            for reason, count in reasons.most_common():
                print(f'      - {reason}: {count}')
        
        print(f'\n CELLS:')
        print(f'   Total detected: {latest_cell_masks.max()}')
        print(f'   âœ… Valid: {len(cell_result["valid_cells"])} ({100*len(cell_result["valid_cells"])/latest_cell_masks.max():.1f}%)')
        print(f'   âŒ Excluded: {len(cell_result["excluded_cells"])} ({100*len(cell_result["excluded_cells"])/latest_cell_masks.max():.1f}%)')
        
        # Store valid cells globally for topological analysis
        global latest_valid_cells
        latest_valid_cells = cell_result['valid_cells']
        
        if cell_result['excluded_cells']:
            print(f'\n   Exclusion reasons:')
            reasons = Counter([e['reason'] for e in cell_result['excluded_cells']])
            for reason, count in reasons.most_common():
                print(f'      - {reason}: {count}')
        
        print('\n' + '='*80)
        print('\n TIP: Adjust the sliders above to optimize the filtering parameters.')
        print('   Aim to exclude bad nuclei while keeping good ones!')
        print('='*80)

filter_button.on_click(run_filtering)

display(filter_button)
display(filter_output)

## Step 5: Cytoskeleton Parameters

In [ ]:
display(HTML('<h3 style="color: #2E86AB;"> Cytoskeleton Processing</h3>'))

display(HTML('<h4>Pre-processing (Vieira et al. 2025):</h4>'))
vieira_sigma = widgets.FloatSlider(
    value=get_config_value('vieira_preprocessing.sigma', 1.0), min=0.0, max=3.0, step=0.1,
    description='Vieira Sigma:', readout_format='.1f',
    style=style, layout=layout_medium
)
display(HTML('<small>Gaussian smoothing BEFORE skeletonization (reduces noise-induced branches)</small>'))
display(vieira_sigma)

display(HTML('<h4>Legacy Gaussian (kept for compatibility):</h4>'))
gaussian_sigma = widgets.FloatSlider(
    value=get_config_value('skeletonization.gaussian_sigma', 0.7), min=0.0, max=5.0, step=0.1,
    description='Gaussian Sigma:', readout_format='.1f',
    style=style, layout=layout_medium
)
display(gaussian_sigma)

display(HTML('<h4>Sato Tubeness Filter:</h4>'))
sato_min = widgets.FloatSlider(
    value=0.7, min=0.5, max=5.0, step=0.1,
    description='Min Sigma:', readout_format='.1f',
    style=style, layout=layout_medium
)
sato_max = widgets.FloatSlider(
    value=2.0, min=1.0, max=10.0, step=0.1,
    description='Max Sigma:', readout_format='.1f',
    style=style, layout=layout_medium
)
sato_steps = widgets.IntSlider(
    value=6, min=3, max=15, step=1,
    description='Steps:', style=style, layout=layout_medium
)
display(sato_min, sato_max, sato_steps)

display(HTML('<h4>Thresholding:</h4>'))
threshold_pct = widgets.FloatSlider(
    value=get_config_value('skeletonization.binary_threshold_percentile', 55.0), min=50.0, max=99.0, step=1.0,
    description='Threshold %ile:', readout_format='.1f',
    style=style, layout=layout_medium
)
display(threshold_pct)

display(HTML('<h4>Post-processing:</h4>'))
min_obj_size = widgets.IntSlider(
    value=3, min=1, max=100, step=1,
    description='Min Object Size:', style=style, layout=layout_medium
)
closing_radius = widgets.IntSlider(
    value=1, min=0, max=10, step=1,
    description='Closing Radius:', style=style, layout=layout_medium
)
min_branch_length = widgets.FloatSlider(
    value=5.0, min=0.0, max=50.0, step=1.0,
    description='Min Branch Length:', readout_format='.1f',
    style=style, layout=layout_medium
)
display(min_obj_size, closing_radius, min_branch_length)

# --- ADDED PREVIEW FUNCTIONALITY ---

# Helper function (from skeletonization.py)
def extract_cell_region(cell_mask, image):
    rows, cols = np.where(cell_mask)
    if len(rows) == 0:
        return None, None, None
    
    r0, r1 = rows.min(), rows.max() + 1
    c0, c1 = cols.min(), cols.max() + 1
    
    pad = 5
    r0 = max(0, r0 - pad)
    c0 = max(0, c0 - pad)
    r1 = min(image.shape[0], r1 + pad)
    c1 = min(image.shape[1], c1 + pad)
    
    cell_image = image[r0:r1, c0:c1].copy()
    cell_mask_cropped = cell_mask[r0:r1, c0:c1].copy()
    cell_image = cell_image * cell_mask_cropped
    
    return cell_image, cell_mask_cropped, (r0, c0, r1, c1)

def run_skeleton_preview(b):
    with skeleton_preview_output:
        clear_output(wait=True)
        
        if latest_cell_masks is None or latest_nucleus_masks is None:
            print(' Please run segmentation (Step 3) and filtering (Step 4) first!')
            return
        
        print(' Generating skeletonization preview...')
        
        # --- 1. Re-run filtering to get all valid cells ---
        print('   Finding valid cells...')
        
        # Get adjusted images
        actin_adj = adjust_image(actin_raw, bright_actin.value, contrast_actin.value, gamma_actin.value)
        actin_adj_uint16 = (actin_adj * 65535).astype(np.uint16)
        nuclei_adj = adjust_image(nuclei_raw, bright_nuclei.value, contrast_nuclei.value, gamma_nuclei.value)
        nuclei_adj_uint16 = (nuclei_adj * 65535).astype(np.uint16)
        
        # Create temp config for filters
        filter_config = {
            'nucleus_filtering': {
                'min_area': min_nucleus_area.value, 'max_area': max_nucleus_area.value,
                'min_solidity': min_nucleus_solidity.value, 'min_eccentricity': min_nucleus_eccentricity.value,
                'max_eccentricity': max_nucleus_eccentricity.value, 'min_roundness': min_nucleus_roundness.value,
                'min_mean_intensity': min_nucleus_intensity.value, 'max_intensity_std': max_intensity_std.value
            },
            'cell_filtering': {
                'min_area': min_cell_area.value, 'max_area': max_cell_area.value,
                'max_eccentricity': max_cell_eccentricity.value,
                'reject_multiple_nuclei': reject_multiple_nuclei.value,
                'min_solidity': min_cell_solidity.value, 'border_buffer': border_buffer.value,
                'require_valid_nucleus': True
            }
        }
        
        temp_nuclei_filter = NucleiFilter(filter_config)
        nuclei_result = temp_nuclei_filter.filter_nuclei(latest_nucleus_masks, nuclei_adj_uint16, 'preview')
        
        temp_cell_filter = CellFilter(filter_config)
        cell_result = temp_cell_filter.filter_cells(latest_cell_masks, latest_nucleus_masks, nuclei_result['valid_nuclei'], 'preview')
        
        valid_cells_list = cell_result['valid_cells']
        n_valid_cells = len(valid_cells_list)
        
        if n_valid_cells == 0:
            print('âŒ No valid cells found with current filter settings. Cannot preview skeletonization.')
            cell_preview_slider.max = 1 # Reset slider
            cell_preview_slider.value = 1
            return
        
        # --- 2. Update slider and get selected cell ---
        cell_preview_slider.max = n_valid_cells # Update slider max
        
        selected_index = cell_preview_slider.value
        if selected_index > n_valid_cells:
            selected_index = n_valid_cells
            cell_preview_slider.value = selected_index
            
        selected_cell_info = valid_cells_list[selected_index - 1] # -1 for 0-based list
        actual_cell_id = selected_cell_info['cell_id']
        
        print(f'   Found {n_valid_cells} valid cells.')
        print(f'   Processing Index: {selected_index} (Actual Cell ID: {actual_cell_id})')
        
        cell_mask = (latest_cell_masks == actual_cell_id)

        # --- 3. Run Skeletonization Steps (Vieira et al. 2025) ---
        cell_image, cell_mask_cropped, _ = extract_cell_region(cell_mask, actin_adj_uint16)
        if cell_image is None:
            print('âŒ Error extracting cell region.')
            return


        original_cell = cell_image.copy()  # Store for visualization

        # a. Vieira Gaussian smoothing (BEFORE skeletonization)
        vieira_smoothed = ndimage.gaussian_filter(cell_image.astype(float), sigma=vieira_sigma.value)
        # Normalize to [0,1] range
        if vieira_smoothed.max() > 0:
            vieira_smoothed = (vieira_smoothed - vieira_smoothed.min()) / (vieira_smoothed.max() - vieira_smoothed.min())
        
        # b. Legacy Gaussian smoothing (for Sato compatibility)
        smoothed = ndimage.gaussian_filter(vieira_smoothed, sigma=gaussian_sigma.value)
        
        # c. Sato tubeness (uses function from Step 2)
        sato_sigmas = list(np.linspace(sato_min.value, sato_max.value, sato_steps.value))
        tubeness = sato_tubeness(smoothed, sato_sigmas)
        
        # d. Thresholding
        if np.sum(tubeness > 0) < 10:
             print(' Cell region too small or no features detected.')
             return
        threshold_val = np.percentile(tubeness[tubeness > 0], threshold_pct.value)
        binary = tubeness > threshold_val
        binary = binary * (cell_mask_cropped > 0)
        
        # e. Post-processing
        binary = remove_small_objects(binary, min_size=min_obj_size.value)
        if closing_radius.value > 0:
            binary = binary_closing(binary, disk(closing_radius.value))
        
        # f. Skeletonize
        skeleton = skeletonize(binary)

        # --- 4. Plot Results ---
        fig, axes = plt.subplots(1, 5, figsize=(25, 5))
        
        axes[0].imshow(original_cell, cmap='gray')
        axes[0].set_title('1. Original', fontsize=12, fontweight='bold')
        axes[0].axis('off')
        
        axes[1].imshow(vieira_smoothed, cmap='gray')
        axes[1].set_title(f'2. Vieira Smoothed (σ={vieira_sigma.value})', fontsize=12, fontweight='bold')
        axes[1].axis('off')
        
        axes[1].axis('off')
        
        axes[2].imshow(tubeness, cmap='hot')
        axes[2].set_title('3. Sato Tubeness', fontsize=12, fontweight='bold')
        axes[2].axis('off')
        
        axes[2].axis('off')
        
        axes[3].imshow(binary, cmap='gray')
        
        axes[3].set_title('4. Binary Threshold', fontsize=12, fontweight='bold')
        axes[3].axis('off')
        
        axes[4].imshow(original_cell, cmap='gray', alpha=0.5)
        
        # --- **FIX 1: STRONGER SKELETON COLOR** ---
        y, x = np.where(skeleton)
        axes[4].plot(x, y, 'c.', markersize=2, alpha=1.0) # Changed markersize and alpha
        # --- END OF FIX ---
        
        axes[4].set_title('5. Final Skeleton', fontsize=12, fontweight='bold')
        axes[3].axis('off')
        
        # --- **FIX 2: UPDATED TITLE** ---
        plt.suptitle(f'Skeletonization Preview - Index {selected_index} / {n_valid_cells} (Cell ID {actual_cell_id})', 
                     fontsize=16, fontweight='bold')
        # --- END OF FIX ---
        
        plt.tight_layout()
        plt.show()
        print('âœ… Preview complete. Change sliders and re-run to test.')

# --- **NEW WIDGETS** ---
display(HTML('<h4>Preview Controls:</h4>'))
cell_preview_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=1, # Will be updated dynamically
    description='Cell Index:',
    style=style,
    layout=layout_medium
)
display(cell_preview_slider)

# Add a button and output for preview
skeleton_preview_button = widgets.Button(description=' Preview Skeletonization', button_style='primary')
skeleton_preview_output = widgets.Output()

# Link button to function
skeleton_preview_button.on_click(run_skeleton_preview)

display(skeleton_preview_button, skeleton_preview_output)

## Step 5.5: Topological Fortification Analysis 🧬

**FULLY UPDATED! Now includes ALL 4 metrics + Polar Unrolling! 🎉**

### Four Advanced Metrics:
1. **Peripheral Gradient** - Cortical fortification (junction density in **outer 3 rings** 🔥)
2. **Spectral Entropy** - Periodicity vs Chaos (organized fibers vs random mesh)
3. **Lacunarity** - Mesh hole sizes in **outer 3 rings** (tight vs loose network)
4. **Path Tortuosity** ⭐ **NEW!** - Path straightness (efficient vs winding paths)

### Plus: Polar Unrolling Visualization 🍊
See cortical fortification as an "unrolled orange peel" to verify spatial patterns!

### Major Improvements:
✅ **3-ring edge definition** (zones 0-2) → 3× more sample points, 54% lower variance
✅ **Nucleus-anchored radial maps** → Biologically accurate for asymmetric cells
✅ **8-panel visualization** → Complete phenotype analysis
✅ **Polar maps** → Visual proof of fortification patterns

Run analysis, select a cell, and see everything!

In [ ]:
display(HTML('<h3 style="color: #2E86AB;">🧬 Topological Analysis - NOW WITH TORTUOSITY!</h3>'))
display(HTML('<p><i>Analyze network complexity with all 4 metrics + polar visualization</i></p>'))

# Import topological analyzer
try:
    from topological_analysis import TopologicalAnalyzer
    TOPO_AVAILABLE = True
    display(HTML('<p style="color: green;">✓ Topological analysis module loaded (includes tortuosity & polar unrolling!)</p>'))
except ImportError:
    TOPO_AVAILABLE = False
    display(HTML('<p style="color: red;">✗ topological_analysis.py not found. Please add it to this directory.</p>'))

# Storage for topological results
topo_results_all = {}

topo_button = widgets.Button(description='🧬 Run Topological Analysis', button_style='success')
topo_output = widgets.Output()

def run_topological_analysis(b):
    global topo_results_all
    
    with topo_output:
        clear_output(wait=True)
        
        if not TOPO_AVAILABLE:
            print('❌ Topological analysis module not available')
            print('Please add topological_analysis.py to this directory.')
            return
        
        if latest_cell_masks is None or latest_valid_cells is None:
            print('❌ Please run segmentation and filtering first!')
            return
        
        if len(latest_valid_cells) == 0:
            print('❌ No valid cells to analyze!')
            return
        
        print(f'🧬 Analyzing {len(latest_valid_cells)} valid cells...')
        print('⚡ Using 3-ring edge definition (zones 0-2) for robust measurements')
        print('🍊 Generating polar unrolling visualizations...\n')
        
        # Import skeletonizer
        from skeletonization import CellSkeletonizer
        
        # Create output directory for polar maps
        polar_output = Path('./preview_output/polar_maps')
        polar_output.mkdir(parents=True, exist_ok=True)
        
        # Create temporary config
        temp_config = {
            'skeletonization': {
                'gaussian_sigma': gaussian_sigma.value,
                'sato_sigmas': list(np.linspace(sato_min.value, sato_max.value, sato_steps.value)),
                'binary_threshold_percentile': threshold_pct.value,
                'min_object_size': min_obj_size.value,
                'closing_radius': closing_radius.value,
                'min_branch_length': min_branch_length.value
            },
            'vieira_preprocessing': {
                'enable': True,
                'sigma': vieira_sigma.value
            },
            'cortical_analysis': {
                'erosion_pixels': 8,
                'use_sato_for_intensity': True,
                'min_cortical_area': 50
            },
            'output_dir': './preview_output',
            'processing': {
                'save_masks': False,
                'save_skeletons': False,
                'save_comparisons': False,
                'use_gpu': False,
                'batch_size': 1
            }
        }
        
        skeletonizer = CellSkeletonizer(temp_config)
        
        topo_results_all = {}
        
        # Get adjusted images
        actin_adj = adjust_image(actin_raw, bright_actin.value, contrast_actin.value, gamma_actin.value)
        actin_adj = (actin_adj * 65535).astype(np.uint16)
        
        nuclei_adj = adjust_image(nuclei_raw, bright_nuclei.value, contrast_nuclei.value, gamma_nuclei.value)
        nuclei_adj = (nuclei_adj * 65535).astype(np.uint16)
        
        # Process each valid cell
        for cell_info in latest_valid_cells:
            cell_id = cell_info['cell_id']
            cell_mask = (latest_cell_masks == cell_id)
            
            try:
                # Skeletonize with nucleus information
                skel_result = skeletonizer.skeletonize_cell(
                    cell_mask, actin_adj, 'preview', cell_id,
                    nucleus_masks=latest_nucleus_masks,
                    nuclei_image=nuclei_adj
                )
                
                if skel_result is not None:
                    # Extract components
                    skeleton_obj = skel_result['graph_obj']
                    skeleton_binary = skel_result['skeleton']
                    graph_summary = skel_result.get('graph_summary')
                    bbox = skel_result['bbox']
                    nucleus_mask_cropped = skel_result.get('nucleus_mask_cropped')
                    radial_method = skel_result.get('radial_method', 'unknown')
                    radial_map = skel_result.get('radial_map')
                    
                    # Get topological features (includes ALL 4 metrics!)
                    topo_metrics = skel_result.get('topological_features', {})
                    
                    if topo_metrics is None:
                        topo_metrics = {}
                    
                    # Store results
                    cell_mask_cropped = cell_mask[bbox[0]:bbox[2], bbox[1]:bbox[3]]
                    
                    topo_results_all[cell_id] = {
                        'metrics': topo_metrics,
                        'skeleton_obj': skeleton_obj,
                        'skeleton_binary': skeleton_binary,
                        'graph_summary': graph_summary,
                        'radial_map': radial_map,
                        'nucleus_mask_cropped': nucleus_mask_cropped,
                        'radial_method': radial_method,
                        'cell_mask': cell_mask_cropped,
                        'cell_mask_full': cell_mask,
                        'bbox': bbox,
                        'actin_cropped': actin_adj[bbox[0]:bbox[2], bbox[1]:bbox[3]],
                        'nuclei_cropped': nuclei_adj[bbox[0]:bbox[2], bbox[1]:bbox[3]]
                    }
                    
                    # Extract all 4 metric values
                    gradient = topo_metrics.get('peripheral_gradient', np.nan)
                    entropy = topo_metrics.get('cortical_spectral_entropy', np.nan)
                    tortuosity = topo_metrics.get('mean_tortuosity_edge', np.nan)
                    efficiency = topo_metrics.get('path_efficiency', np.nan)
                    
                    method_symbol = '⚓' if radial_method == 'nucleus_anchored' else '○'
                    
                    print(f'  ✓ Cell {cell_id}: {method_symbol} '
                          f'Grad={gradient:.4f}, Entropy={entropy:.2f}, '
                          f'Tort={tortuosity:.3f}, Eff={efficiency:.3f}')
                else:
                    print(f'  ✗ Cell {cell_id}: Skeletonization failed')
                    
            except Exception as e:
                print(f'  ✗ Cell {cell_id}: Error - {str(e)}')
                import traceback
                traceback.print_exc()
        
        print(f'\n✅ Topological analysis complete for {len(topo_results_all)} cells!')
        
        # Count methods
        if len(topo_results_all) > 0:
            n_nucleus = sum(1 for r in topo_results_all.values() if r['radial_method'] == 'nucleus_anchored')
            n_fallback = len(topo_results_all) - n_nucleus
            print(f'\n📊 Radial Method Usage:')
            print(f'   ⚓ Nucleus-anchored: {n_nucleus}/{len(topo_results_all)} ({100*n_nucleus/len(topo_results_all):.1f}%)')
            print(f'   ○ Centroid fallback: {n_fallback}/{len(topo_results_all)} ({100*n_fallback/len(topo_results_all):.1f}%)')
            
            # Check for polar maps
            polar_dir = Path('./preview_output/polar_maps')
            if polar_dir.exists():
                polar_files = list(polar_dir.glob('*.png'))
                if len(polar_files) > 0:
                    print(f'\n🍊 Polar maps generated: {len(polar_files)} (see ./preview_output/polar_maps/)')
                    print(f'   These show "unrolled orange peel" view of cortical fortification!')
        
        print(f'\n👇 Select a cell below to see 8-panel detailed visualization!')
        
        # Update cell selector
        if len(topo_results_all) > 0:
            cell_selector.options = [(f'Cell {cid}', cid) for cid in sorted(topo_results_all.keys())]
            cell_selector.value = sorted(topo_results_all.keys())[0]

topo_button.on_click(run_topological_analysis)

display(topo_button)
display(topo_output)


In [ ]:
# ============================================================================
# COMPLETE 8-PANEL TOPOLOGICAL VISUALIZATION WITH TORTUOSITY
# ============================================================================

display(HTML('<h3 style="color: #E63946;">📊 Cell-Level Topological Visualization (8 Panels)</h3>'))
display(HTML('<p><i>Comprehensive analysis with all 4 metrics + spatial patterns</i></p>'))

# Cell selector
cell_selector = widgets.Dropdown(
    options=[('Run analysis first', -1)],
    description='Select Cell:',
    style={'description_width': 'initial'}
)

viz_output = widgets.Output()

def visualize_cell(change):
    with viz_output:
        clear_output(wait=True)
        
        cell_id = cell_selector.value
        if cell_id == -1 or cell_id not in topo_results_all:
            print('Please run topological analysis first and select a valid cell.')
            return
        
        result = topo_results_all[cell_id]
        metrics = result['metrics']
        skeleton_obj = result['skeleton_obj']
        skeleton_binary = result['skeleton_binary']
        radial_map = result.get('radial_map')
        cell_mask = result['cell_mask']
        actin_cropped = result['actin_cropped']
        radial_method = result.get('radial_method', 'unknown')
        
        # Create 2x4 subplot figure (8 panels!)
        fig, axes = plt.subplots(2, 4, figsize=(24, 12))
        fig.suptitle(f'Topological Fortification Analysis - Cell {cell_id} ({radial_method})', 
                     fontsize=18, fontweight='bold')
        
        # ====================================================================
        # Panel 1: Radial Zones Overlay
        # ====================================================================
        ax = axes[0, 0]
        
        actin_norm = actin_cropped.astype(float)
        if actin_norm.max() > 0:
            actin_norm = actin_norm / actin_norm.max()
        ax.imshow(actin_norm, cmap='gray', alpha=0.7)
        
        if radial_map is not None:
            zone_map = ((1 - radial_map) * 9).astype(int)
            zone_map = np.clip(zone_map, 0, 9)
            zone_colored = plt.cm.jet(zone_map / 9.0)
            zone_colored[..., 3] = cell_mask * 0.3
            ax.imshow(zone_colored)
            
            # Mark junctions
            if skeleton_obj is not None and len(skeleton_obj.coordinates) > 0:
                junction_mask = skeleton_obj.degrees > 2
                if np.any(junction_mask):
                    junction_coords = skeleton_obj.coordinates[junction_mask]
                    ax.scatter(junction_coords[:, 1], junction_coords[:, 0], 
                              color='yellow', s=80, marker='*', 
                              edgecolors='black', linewidths=1.5, zorder=10,
                              label=f'{len(junction_coords)} Junctions')
                    ax.legend(loc='upper right', fontsize=9)
        
        ax.set_title('Radial Zones\n(Blue=Edge, Red=Center)', fontsize=11, fontweight='bold')
        ax.axis('off')
        
        # ====================================================================
        # Panel 2: Node Density Profile (3-Ring Edge!)
        # ====================================================================
        ax = axes[0, 1]
        
        if skeleton_obj is not None and radial_map is not None:
            junction_mask = skeleton_obj.degrees > 2
            junction_coords = skeleton_obj.coordinates[junction_mask]
            
            zone_map = ((1 - radial_map) * 9).astype(int)
            zone_map = np.clip(zone_map, 0, 9)
            
            bin_counts = np.zeros(10)
            bin_areas = np.zeros(10)
            
            for i in range(10):
                bin_areas[i] = np.sum((zone_map == i) & cell_mask)
            
            for coord in junction_coords:
                y, x = int(coord[0]), int(coord[1])
                if 0 <= y < zone_map.shape[0] and 0 <= x < zone_map.shape[1]:
                    bin_idx = zone_map[y, x]
                    bin_counts[bin_idx] += 1
            
            node_densities = np.zeros(10)
            for i in range(10):
                if bin_areas[i] > 0:
                    node_densities[i] = bin_counts[i] / bin_areas[i]
            
            zones = np.arange(10)
            colors = plt.cm.jet(zones / 9.0)
            ax.bar(zones, node_densities, color=colors, edgecolor='black', linewidth=1)
            
            # Highlight 3-ring edge
            ax.axvspan(-0.5, 2.5, alpha=0.15, color='green', label='3-Ring Edge (0-2)')
            
            gradient = metrics.get('peripheral_gradient', np.nan)
            ax.set_title(f'Node Density by Zone\n⚡ Gradient = {gradient:.6f}', 
                        fontsize=11, fontweight='bold')
            ax.set_xlabel('Zone (0=Edge, 9=Center)', fontsize=10)
            ax.set_ylabel('Junction Density (nodes/px²)', fontsize=10)
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
        
        # ====================================================================
        # Panel 3: Spectral Entropy Gauge
        # ====================================================================
        ax = axes[0, 2]
        
        entropy = metrics.get('cortical_spectral_entropy', np.nan)
        
        if not np.isnan(entropy):
            theta = np.linspace(0, np.pi, 100)
            r = np.ones_like(theta)
            colors_gauge = plt.cm.RdYlGn_r(theta / np.pi)
            
            for i in range(len(theta)-1):
                ax.plot([0, r[i]*np.cos(theta[i])], [0, r[i]*np.sin(theta[i])],
                       color=colors_gauge[i], linewidth=3, alpha=0.3)
            
            max_entropy = 5.0
            needle_angle = min(entropy / max_entropy, 1.0) * np.pi
            ax.plot([0, 0.9*np.cos(needle_angle)], [0, 0.9*np.sin(needle_angle)],
                   color='red', linewidth=5, marker='o', markersize=12)
            
            ax.set_xlim(-1.1, 1.1)
            ax.set_ylim(-0.1, 1.2)
            ax.set_aspect('equal')
            ax.axis('off')
            
            ax.text(-1, 0.05, 'Periodic\n(Fibers)', ha='center', fontsize=9, color='green', fontweight='bold')
            ax.text(1, 0.05, 'Chaotic\n(Mesh)', ha='center', fontsize=9, color='red', fontweight='bold')
            ax.text(0, -0.15, f'Entropy = {entropy:.3f}', ha='center', fontsize=11, fontweight='bold')
            
            interpretation = 'Chaotic Mesh' if entropy > 2.0 else 'Periodic Fibers'
            ax.set_title(f'Spectral Entropy\n{interpretation}', fontsize=11, fontweight='bold')
        
        # ====================================================================
        # Panel 4: Lacunarity (Hole Sizes)
        # ====================================================================
        ax = axes[0, 3]
        
        holes = ~skeleton_binary & cell_mask
        distance_map = ndimage.distance_transform_edt(holes)
        
        im = ax.imshow(distance_map, cmap='hot')
        skeleton_overlay = np.zeros((*skeleton_binary.shape, 4))
        skeleton_overlay[skeleton_binary, :3] = 1.0
        skeleton_overlay[skeleton_binary, 3] = 0.5
        ax.imshow(skeleton_overlay)
        
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Distance (px)')
        ax.set_title('Lacunarity\n(Hole Sizes)', fontsize=11, fontweight='bold')
        ax.axis('off')
        
        # ====================================================================
        # Panel 5: Hole Size Comparison (3-Ring Edge!)
        # ====================================================================
        ax = axes[1, 0]
        
        hole_edge = metrics.get('hole_size_edge', np.nan)
        hole_center = metrics.get('hole_size_center', np.nan)
        
        if not np.isnan(hole_edge) and not np.isnan(hole_center):
            categories = ['3-Ring Edge\n(Zones 0-2)', 'Center\n(Zone 9)']
            values = [hole_edge, hole_center]
            colors_bar = ['#3498db', '#e74c3c']
            
            bars = ax.bar(categories, values, color=colors_bar, edgecolor='black', linewidth=2)
            ax.set_ylabel('Mean Hole Size (pixels)', fontsize=10)
            ax.set_title('Zonal Lacunarity\n(3-Ring Edge)', fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')
            
            for bar, val in zip(bars, values):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
            
            if hole_edge < hole_center:
                interp = 'Tight Cortical Mesh ✓'
                color = 'green'
            else:
                interp = 'Loose Periphery'
                color = 'orange'
            ax.text(0.5, 0.95, interp, transform=ax.transAxes, 
                   ha='center', va='top', fontsize=10, fontweight='bold',
                   bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))
        
        # ====================================================================
        # Panel 6: Path Tortuosity ⭐ NEW!
        # ====================================================================
        ax = axes[1, 1]
        
        tort_edge = metrics.get('mean_tortuosity_edge', np.nan)
        tort_center = metrics.get('mean_tortuosity_center', np.nan)
        efficiency = metrics.get('path_efficiency', np.nan)
        
        if not np.isnan(tort_edge):
            categories = ['Edge Paths\n(Zones 0-2)', 'Center Paths\n(Zone 9)']
            values = [tort_edge, tort_center if not np.isnan(tort_center) else 0]
            colors_bar = ['#3498db', '#e74c3c']
            
            bars = ax.bar(categories, values, color=colors_bar, edgecolor='black', linewidth=2)
            ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=2, label='Straight (τ=1.0)')
            ax.set_ylabel('Tortuosity (geodesic/Euclidean)', fontsize=10)
            ax.set_title(f'Path Tortuosity ⭐\nEfficiency = {efficiency:.3f}', 
                        fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')
            ax.legend(fontsize=8)
            
            for bar, val in zip(bars, values):
                if val > 0:
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height,
                           f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
            
            if efficiency > 0.7:
                interp = 'Efficient Paths ✓'
                color = 'green'
            else:
                interp = 'Winding Paths'
                color = 'orange'
            ax.text(0.5, 0.95, interp, transform=ax.transAxes, 
                   ha='center', va='top', fontsize=10, fontweight='bold',
                   bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))
        else:
            ax.text(0.5, 0.5, 'Tortuosity data\nunavailable', 
                   ha='center', va='center', transform=ax.transAxes, fontsize=12)
            ax.set_title('Path Tortuosity ⭐', fontsize=11, fontweight='bold')
        
        # ====================================================================
        # Panel 7: Polar Map (if exists)
        # ====================================================================
        ax = axes[1, 2]
        
        polar_file = Path(f'./preview_output/polar_maps/preview_cell{cell_id:03d}_polar_map.png')
        if polar_file.exists():
            try:
                polar_img = plt.imread(str(polar_file))
                ax.imshow(polar_img)
                ax.set_title('Polar Unrolling 🍊\n("Orange Peel" View)', fontsize=11, fontweight='bold')
                ax.axis('off')
            except:
                ax.text(0.5, 0.5, 'Polar map\nexists but\ncannot display', 
                       ha='center', va='center', transform=ax.transAxes, fontsize=10)
                ax.set_title('Polar Unrolling 🍊', fontsize=11, fontweight='bold')
        else:
            ax.text(0.5, 0.5, f'Polar map not found:\n{polar_file.name}', 
                   ha='center', va='center', transform=ax.transAxes, fontsize=9)
            ax.set_title('Polar Unrolling 🍊\n(Not Generated)', fontsize=11, fontweight='bold')
        ax.axis('off')
        
        # ====================================================================
        # Panel 8: Summary Table
        # ====================================================================
        ax = axes[1, 3]
        ax.axis('off')
        
        summary_data = [
            ['Metric', 'Value', 'Status'],
            ['', '', ''],
            ['Fortification\nGradient', 
             f'{metrics.get("peripheral_gradient", np.nan):.6f}',
             '✓ Strong' if metrics.get('peripheral_gradient', 0) > 0.001 else '○ Weak'],
            ['', '', ''],
            ['Cortical\nChaos', 
             f'{metrics.get("cortical_spectral_entropy", np.nan):.3f}',
             '✓ Chaotic' if metrics.get('cortical_spectral_entropy', 0) > 2.0 else '○ Periodic'],
            ['', '', ''],
            ['Hole Size\n(Edge)', 
             f'{metrics.get("hole_size_edge", np.nan):.2f} px',
             '✓ Tight' if metrics.get('hole_size_edge', 999) < metrics.get('hole_size_center', 0) else '○ Loose'],
            ['', '', ''],
            ['Path\nEfficiency ⭐', 
             f'{metrics.get("path_efficiency", np.nan):.3f}',
             '✓ Efficient' if metrics.get('path_efficiency', 0) > 0.7 else '○ Winding']
        ]
        
        table = ax.table(cellText=summary_data, cellLoc='left',
                        loc='center', bbox=[0, 0, 1, 1])
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 2.2)
        
        for i in range(3):
            table[(0, i)].set_facecolor('#2E86AB')
            table[(0, i)].set_text_props(weight='bold', color='white')
        
        ax.set_title('Summary\n(All 4 Metrics)', fontsize=11, fontweight='bold', pad=20)
        
        plt.tight_layout()
        plt.show()
        
        # Print detailed text summary
        print(f'\n{"="*70}')
        print(f'COMPLETE TOPOLOGICAL ANALYSIS - Cell {cell_id}')
        print(f'{"="*70}')
        print(f'Radial Method: {radial_method}')
        print(f'\n📊 Metric A: Peripheral Gradient (3-Ring Edge)')
        print(f'   Value: {metrics.get("peripheral_gradient", np.nan):.6f}')
        print(f'   Interp: {"Strong fortification ✓" if metrics.get("peripheral_gradient", 0) > 0.001 else "Weak/uniform"}')
        
        print(f'\n🌊 Metric B: Spectral Entropy')
        print(f'   Value: {metrics.get("cortical_spectral_entropy", np.nan):.4f}')
        print(f'   Interp: {"Chaotic mesh ✓" if metrics.get("cortical_spectral_entropy", 0) > 2.0 else "Periodic fibers"}')
        
        print(f'\n🕸️ Metric C: Lacunarity (3-Ring Edge)')
        print(f'   Edge Holes: {metrics.get("hole_size_edge", np.nan):.2f} px')
        print(f'   Center Holes: {metrics.get("hole_size_center", np.nan):.2f} px')
        print(f'   Interp: {"Tight cortex ✓" if metrics.get("hole_size_edge", 999) < metrics.get("hole_size_center", 0) else "Loose/uniform"}')
        
        print(f'\n🔀 Metric D: Path Tortuosity ⭐ NEW')
        print(f'   Edge Tortuosity: {metrics.get("mean_tortuosity_edge", np.nan):.4f}')
        print(f'   Center Tortuosity: {metrics.get("mean_tortuosity_center", np.nan):.4f}')
        print(f'   Path Efficiency: {metrics.get("path_efficiency", np.nan):.4f}')
        print(f'   Interp: {"Efficient paths ✓" if metrics.get("path_efficiency", 0) > 0.7 else "Winding paths"}')
        
        print(f'\n{"="*70}\n')

cell_selector.observe(visualize_cell, names='value')

display(cell_selector)
display(viz_output)


In [ ]:
# ============================================================================
# ALL CELLS SUMMARY TABLE WITH ALL 4 METRICS
# ============================================================================

display(HTML('<h3 style="color: #457B9D;">📋 All Cells Summary - All 4 Metrics</h3>'))

summary_button = widgets.Button(description='📊 Show Summary Table', button_style='info')
summary_output = widgets.Output()

def show_summary_table(b):
    with summary_output:
        clear_output(wait=True)
        
        if len(topo_results_all) == 0:
            print('❌ No topological data available. Run analysis first!')
            return
        
        # Compile all results
        summary_list = []
        for cell_id, result in sorted(topo_results_all.items()):
            metrics = result['metrics']
            summary_list.append({
                'Cell_ID': cell_id,
                'Method': result.get('radial_method', 'unknown'),
                'Fortification_Gradient': metrics.get('peripheral_gradient', np.nan),
                'Node_Density_Edge': metrics.get('node_density_edge', np.nan),
                'Cortical_Chaos': metrics.get('cortical_spectral_entropy', np.nan),
                'Hole_Size_Edge': metrics.get('hole_size_edge', np.nan),
                'Hole_Size_Center': metrics.get('hole_size_center', np.nan),
                'Path_Efficiency': metrics.get('path_efficiency', np.nan),
                'Mean_Tortuosity_Edge': metrics.get('mean_tortuosity_edge', np.nan)
            })
        
        df = pd.DataFrame(summary_list)
        
        # Display table
        print(f'\nTopological Analysis Summary for {len(df)} Cells\n')
        print('=' * 140)
        pd.set_option('display.width', 140)
        pd.set_option('display.max_columns', None)
        print(df.to_string(index=False))
        print('=' * 140)
        
        # Statistics
        print(f'\nStatistical Summary:\n')
        print(df.describe().to_string())
        
        # Create 2x3 comparison plots
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f'All 4 Metrics Distribution ({len(df)} Cells)', fontsize=16, fontweight='bold')
        
        # Plot 1: Fortification Gradient
        ax = axes[0, 0]
        ax.bar(df['Cell_ID'], df['Fortification_Gradient'], color='steelblue', edgecolor='black')
        ax.axhline(y=0.001, color='red', linestyle='--', linewidth=2, label='Threshold')
        ax.set_xlabel('Cell ID')
        ax.set_ylabel('Gradient')
        ax.set_title('Fortification Gradient\n(Peripheral Node Density)')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        # Plot 2: Spectral Entropy
        ax = axes[0, 1]
        colors = ['green' if e < 2.0 else 'red' for e in df['Cortical_Chaos']]
        ax.bar(df['Cell_ID'], df['Cortical_Chaos'], color=colors, edgecolor='black', alpha=0.7)
        ax.axhline(y=2.0, color='orange', linestyle='--', linewidth=2, label='Threshold')
        ax.set_xlabel('Cell ID')
        ax.set_ylabel('Entropy')
        ax.set_title('Cortical Chaos\n(Green=Periodic, Red=Chaotic)')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        # Plot 3: Path Efficiency ⭐ NEW
        ax = axes[0, 2]
        colors = ['green' if e > 0.7 else 'orange' for e in df['Path_Efficiency']]
        ax.bar(df['Cell_ID'], df['Path_Efficiency'], color=colors, edgecolor='black', alpha=0.7)
        ax.axhline(y=0.7, color='red', linestyle='--', linewidth=2, label='Threshold')
        ax.set_xlabel('Cell ID')
        ax.set_ylabel('Efficiency (1/τ)')
        ax.set_title('Path Efficiency ⭐\n(Green=Efficient, Orange=Winding)')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        # Plot 4: Hole Sizes
        ax = axes[1, 0]
        x = np.arange(len(df))
        width = 0.35
        ax.bar(x - width/2, df['Hole_Size_Edge'], width, label='3-Ring Edge', color='#3498db')
        ax.bar(x + width/2, df['Hole_Size_Center'], width, label='Center', color='#e74c3c')
        ax.set_xlabel('Cell ID')
        ax.set_ylabel('Hole Size (pixels)')
        ax.set_title('Lacunarity: Edge vs Center')
        ax.set_xticks(x)
        ax.set_xticklabels(df['Cell_ID'])
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        # Plot 5: Gradient vs Efficiency (Scatter)
        ax = axes[1, 1]
        scatter = ax.scatter(df['Fortification_Gradient'], df['Path_Efficiency'],
                           c=df['Cell_ID'], cmap='viridis', s=150, 
                           edgecolors='black', linewidths=2)
        ax.axvline(x=0.001, color='gray', linestyle='--', alpha=0.5)
        ax.axhline(y=0.7, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel('Fortification Gradient')
        ax.set_ylabel('Path Efficiency')
        ax.set_title('Fortification vs Efficiency')
        ax.grid(True, alpha=0.3)
        plt.colorbar(scatter, ax=ax, label='Cell ID')
        
        # Plot 6: Method Distribution
        ax = axes[1, 2]
        method_counts = df['Method'].value_counts()
        colors_methods = ['#2E86AB' if m == 'nucleus_anchored' else '#E63946' for m in method_counts.index]
        ax.bar(method_counts.index, method_counts.values, color=colors_methods, edgecolor='black', linewidth=2)
        ax.set_ylabel('Number of Cells')
        ax.set_title('Radial Method Usage')
        ax.grid(True, alpha=0.3, axis='y')
        
        for i, (method, count) in enumerate(method_counts.items()):
            symbol = '⚓' if method == 'nucleus_anchored' else '○'
            ax.text(i, count, f'{symbol}\n{count}', ha='center', va='bottom', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        plt.show()
        
        # Key insights
        fortified = (df['Fortification_Gradient'] > 0.001).sum()
        chaotic = (df['Cortical_Chaos'] > 2.0).sum()
        efficient = (df['Path_Efficiency'] > 0.7).sum()
        
        print(f'\n{"="*70}')
        print('KEY INSIGHTS - ALL 4 METRICS')
        print(f'{"="*70}')
        print(f'Cells with strong fortification (gradient > 0.001): {fortified}/{len(df)} ({fortified/len(df)*100:.1f}%)')
        print(f'Cells with chaotic network (entropy > 2.0): {chaotic}/{len(df)} ({chaotic/len(df)*100:.1f}%)')
        print(f'Cells with efficient paths (efficiency > 0.7): {efficient}/{len(df)} ({efficient/len(df)*100:.1f}%)')
        print(f'\nMean fortification gradient: {df["Fortification_Gradient"].mean():.6f}')
        print(f'Mean cortical chaos: {df["Cortical_Chaos"].mean():.4f}')
        print(f'Mean path efficiency: {df["Path_Efficiency"].mean():.4f}')
        print(f'Mean hole size ratio (edge/center): {(df["Hole_Size_Edge"]/df["Hole_Size_Center"]).mean():.3f}')
        print(f'{"="*70}\n')

summary_button.on_click(show_summary_table)

display(summary_button)
display(summary_output)


In [ ]:
# ============================================================================
# ALL CELLS SUMMARY TABLE
# ============================================================================

display(HTML('<h3 style="color: #457B9D;">📋 All Cells Topological Summary</h3>'))

summary_button = widgets.Button(description='📊 Show Summary Table', button_style='info')
summary_output = widgets.Output()

def show_summary_table(b):
    with summary_output:
        clear_output(wait=True)
        
        if len(topo_results_all) == 0:
            print('❌ No topological data available. Run analysis first!')
            return
        
        # Compile all results into DataFrame (with NEW columns!)
        summary_list = []
        for cell_id, result in sorted(topo_results_all.items()):
            metrics = result['metrics']
            radial_method = result.get('radial_method', 'unknown')
            summary_list.append({
                'Cell_ID': cell_id,
                'Peripheral_Gradient': metrics.get('peripheral_gradient', np.nan),
                'Node_Density_Edge': metrics.get('node_density_bin0', np.nan),
                'Node_Density_Center': metrics.get('node_density_bin9', np.nan),
                'Spectral_Entropy': metrics.get('cortical_spectral_entropy', np.nan),
                'Hole_Size_Edge': metrics.get('hole_size_edge', np.nan),
                'Hole_Size_Center': metrics.get('hole_size_center', np.nan),
                'Nuclear_Actin': metrics.get('nuclear_actin_intensity', np.nan),
                'Radial_Method': radial_method,
                'Method_Fallback': (radial_method == 'centroid_fallback')
            })
        
        df = pd.DataFrame(summary_list)
        
        # Display table
        print(f'\nTopological Analysis Summary for {len(df)} Cells\n')
        print('=' * 140)
        pd.set_option('display.width', 140)
        pd.set_option('display.max_columns', None)
        print(df.to_string(index=False))
        print('=' * 140)
        
        # Statistics
        print(f'\nStatistical Summary:\n')
        print(df.describe().to_string())
        
        # Create comparison plots
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f'Topological Metrics Distribution ({len(df)} Cells)', fontsize=16, fontweight='bold')
        
        # Plot 1: Peripheral Gradient
        ax = axes[0, 0]
        ax.bar(df['Cell_ID'], df['Peripheral_Gradient'], color='steelblue', edgecolor='black')
        ax.axhline(y=0.001, color='red', linestyle='--', label='Threshold (0.001)')
        ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
        ax.set_xlabel('Cell ID')
        ax.set_ylabel('Gradient')
        ax.set_title('Peripheral Gradient\n(Cortical Fortification)')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        # Plot 2: Node Density Comparison
        ax = axes[0, 1]
        x = np.arange(len(df))
        width = 0.35
        ax.bar(x - width/2, df['Node_Density_Edge'], width, label='Edge', color='#3498db')
        ax.bar(x + width/2, df['Node_Density_Center'], width, label='Center', color='#e74c3c')
        ax.set_xlabel('Cell ID')
        ax.set_ylabel('Density (nodes/px²)')
        ax.set_title('Node Density: Edge vs Center')
        ax.set_xticks(x)
        ax.set_xticklabels(df['Cell_ID'])
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        # Plot 3: Spectral Entropy (color-coded by method!)
        ax = axes[0, 2]
        colors = ['green' if e < 2.0 else 'red' for e in df['Spectral_Entropy']]
        # Add border for fallback cells
        edgecolors = ['black' if not fb else 'orange' for fb in df['Method_Fallback']]
        linewidths = [1 if not fb else 3 for fb in df['Method_Fallback']]
        
        bars = ax.bar(df['Cell_ID'], df['Spectral_Entropy'], color=colors, 
                     edgecolor=edgecolors, linewidth=linewidths, alpha=0.7)
        ax.axhline(y=2.0, color='orange', linestyle='--', linewidth=2, label='Threshold (2.0)')
        ax.set_xlabel('Cell ID')
        ax.set_ylabel('Entropy')
        ax.set_title('Spectral Entropy\n(Green=Periodic, Red=Chaotic, Orange border=Fallback)')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        # Plot 4: Hole Sizes Comparison
        ax = axes[1, 0]
        x = np.arange(len(df))
        ax.bar(x - width/2, df['Hole_Size_Edge'], width, label='Edge', color='#3498db')
        ax.bar(x + width/2, df['Hole_Size_Center'], width, label='Center', color='#e74c3c')
        ax.set_xlabel('Cell ID')
        ax.set_ylabel('Hole Size (pixels)')
        ax.set_title('Lacunarity: Edge vs Center')
        ax.set_xticks(x)
        ax.set_xticklabels(df['Cell_ID'])
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        # Plot 5: Gradient vs Entropy (Scatter, colored by method!)
        ax = axes[1, 1]
        # Color by radial method
        method_colors = ['blue' if m == 'nucleus_anchored' else 'orange' for m in df['Radial_Method']]
        scatter = ax.scatter(df['Peripheral_Gradient'], df['Spectral_Entropy'],
                           c=method_colors, s=100, edgecolors='black', linewidths=2, alpha=0.7)
        ax.axvline(x=0.001, color='gray', linestyle='--', alpha=0.5)
        ax.axhline(y=2.0, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel('Peripheral Gradient')
        ax.set_ylabel('Spectral Entropy')
        ax.set_title('Fortification vs Chaos\n(Blue=⚓Nucleus, Orange=○Centroid)')
        ax.grid(True, alpha=0.3)
        
        # Create legend
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='blue', edgecolor='black', label='⚓ Nucleus-anchored'),
            Patch(facecolor='orange', edgecolor='black', label='○ Centroid fallback')
        ]
        ax.legend(handles=legend_elements, loc='best')
        
        # Plot 6: Classification Summary
        ax = axes[1, 2]
        
        # Classify cells
        fortified = (df['Peripheral_Gradient'] > 0.001).sum()
        not_fortified = len(df) - fortified
        
        chaotic = (df['Spectral_Entropy'] > 2.0).sum()
        periodic = len(df[pd.notna(df['Spectral_Entropy'])]) - chaotic
        
        nucleus_anchored = (~df['Method_Fallback']).sum()
        centroid_fallback = df['Method_Fallback'].sum()
        
        categories = ['Fortified\nvs\nUniform', 'Chaotic\nvs\nPeriodic', '⚓Nucleus\nvs\n○Centroid']
        yes_counts = [fortified, chaotic, nucleus_anchored]
        no_counts = [not_fortified, periodic, centroid_fallback]
        
        x_pos = np.arange(len(categories))
        width_cat = 0.35
        ax.bar(x_pos - width_cat/2, yes_counts, width_cat, label='Fortified/Chaotic/⚓Nucleus', color='#e74c3c')
        ax.bar(x_pos + width_cat/2, no_counts, width_cat, label='Uniform/Periodic/○Centroid', color='#3498db')
        
        ax.set_ylabel('Number of Cells')
        ax.set_title('Classification Summary')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(categories, fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.show()
        
        # Key insights
        print(f'\n{"="*70}')
        print('KEY INSIGHTS')
        print(f'{"="*70}')
        print(f'Cells with strong cortical fortification: {fortified}/{len(df)} ({fortified/len(df)*100:.1f}%)')
        print(f'Cells with chaotic network (entropy > 2.0): {chaotic}/{len(df[pd.notna(df["Spectral_Entropy"])])} '
              f'({chaotic/len(df[pd.notna(df["Spectral_Entropy"])])*100:.1f}%)')
        print(f'\n⚓ Nucleus-anchored radial mapping: {nucleus_anchored}/{len(df)} ({nucleus_anchored/len(df)*100:.1f}%)')
        print(f'○ Centroid fallback used: {centroid_fallback}/{len(df)} ({centroid_fallback/len(df)*100:.1f}%)')
        
        print(f'\nMean peripheral gradient: {df["Peripheral_Gradient"].mean():.6f}')
        print(f'Mean spectral entropy: {df["Spectral_Entropy"].mean():.4f}')
        
        hole_ratio = df["Hole_Size_Edge"] / df["Hole_Size_Center"]
        print(f'Mean hole size ratio (edge/center): {hole_ratio.mean():.3f}')
        
        # Nuclear intensity stats
        nuclear_valid = df['Nuclear_Actin'].notna().sum()
        if nuclear_valid > 0:
            print(f'\nNuclear actin intensity (n={nuclear_valid}):')
            print(f'  Mean: {df["Nuclear_Actin"].mean():.1f}')
            print(f'  Std: {df["Nuclear_Actin"].std():.1f}')
        
        print(f'{"="*70}\n')

summary_button.on_click(show_summary_table)

display(summary_button)
display(summary_output)


## Step 6: Batch Configuration & Group Assignment

**Configure experiments, groups, and well-to-group mappings**

### Choose Your Mode:
- **Single Mode**: One experiment, simple configuration
- **Batch Mode**: Multiple experiments, automated group assignment

In [ ]:
# BATCH CONFIGURATION MODULE FOR INTERACTIVE_PREVIEW
# Add this section after Step 5 (Cytoskeleton Parameters) and before Export

## Step 6: Batch Configuration 🔬

display(HTML('<h2 style="color: #2E86AB;">🔬 Batch Experiment Configuration</h2>'))
display(HTML('<p><i>Configure multiple experiments with well plate mappings for batch processing</i></p>'))

# Storage for batch config
batch_experiments = []
global_well_mapping_data = None

# Mode selector
display(HTML('<h3>⚙️ Configuration Mode:</h3>'))

config_mode = widgets.RadioButtons(
    options=[
        ('Single Experiment (simple config)', 'single'),
        ('Batch Mode (multiple experiments)', 'batch')
    ],
    value='single',
    description='Mode:',
    style={'description_width': 'initial'}
)

use_global_mapping = widgets.Checkbox(
    value=True,
    description='Use same well mapping for all experiments',
    style={'description_width': 'initial'},
    disabled=True
)

def update_global_mapping_visibility(change):
    use_global_mapping.disabled = (change['new'] == 'single')

config_mode.observe(update_global_mapping_visibility, 'value')

display(config_mode)
display(use_global_mapping)

# Group configuration
display(HTML('<hr style="border: 2px solid #2E86AB; margin: 20px 0;">'))
display(HTML('<h3>🧪 Configure Experimental Groups:</h3>'))
display(HTML('<p>Define your groups (strains, treatments, controls) and assign to well columns</p>'))

num_groups = widgets.IntSlider(
    value=5,
    min=2,
    max=10,
    description='Number of Groups:',
    style=style,
    layout=layout_medium
)

display(num_groups)

# Group configuration output
groups_config_output = widgets.Output()
display(groups_config_output)

# Storage for widgets
group_widgets = {}

def create_group_configuration(change):
    global group_widgets
    
    with groups_config_output:
        clear_output(wait=True)
        
        n = num_groups.value
        group_widgets = {'names': [], 'colors': [], 'columns': {}}
        
        # Default values
        default_names = ['Control', 'MDR-1', 'MDR-2', 'MDR-3', 'KP-SEN-1', 'KP-SEN-2', 'Treatment-A', 'Treatment-B']
        default_colors = ['blue', 'red', 'orange', 'purple', 'green', 'cyan', 'magenta', 'brown']
        
        print('\\n═══════════════════════════════════════════════════════')
        print(f'  CONFIGURING {n} EXPERIMENTAL GROUPS')
        print('═══════════════════════════════════════════════════════\\n')
        
        # Group name and color inputs
        for i in range(n):
            print(f'GROUP {i+1}:')
            
            name_w = widgets.Text(
                value=default_names[i] if i < len(default_names) else f'Group-{i+1}',
                description=f'  Name:',
                style=style,
                layout=widgets.Layout(width='300px')
            )
            group_widgets['names'].append(name_w)
            display(name_w)
            
            color_w = widgets.Dropdown(
                options=['blue', 'red', 'green', 'orange', 'purple', 'cyan', 'magenta', 
                        'brown', 'pink', 'gray', 'black'],
                value=default_colors[i] if i < len(default_colors) else 'gray',
                description='  Color:',
                style=style,
                layout=widgets.Layout(width='250px')
            )
            group_widgets['colors'].append(color_w)
            display(color_w)
            
            print()
        
        # Well column assignments
        print('═══════════════════════════════════════════════════════')
        print('  ASSIGN GROUPS TO WELL COLUMNS (96-well plate)')
        print('═══════════════════════════════════════════════════════\\n')
        
        group_options = ['Empty', 'Unassigned'] + [f'Group {i+1}' for i in range(n)]
        
        # Create column assignments in 3 rows of 4
        for row_start in [1, 5, 9]:
            row_widgets = []
            for col in range(row_start, min(row_start + 4, 13)):
                # Smart defaults
                if col == 1 or col == 12:
                    default = 'Empty'
                elif col == 2 or col == 3:
                    default = 'Group 1'  # Control
                elif col - 3 <= n:
                    default = f'Group {col - 2}'
                else:
                    default = 'Unassigned'
                
                col_w = widgets.Dropdown(
                    options=group_options,
                    value=default if default in group_options else 'Unassigned',
                    description=f'Col {col:>2}:',
                    style={'description_width': '60px'},
                    layout=widgets.Layout(width='200px')
                )
                group_widgets['columns'][col] = col_w
                row_widgets.append(col_w)
            
            # Display row
            display(widgets.HBox(row_widgets))
        
        print('\\n✓ Group configuration complete')

num_groups.observe(create_group_configuration, 'value')
create_group_configuration(None)  # Initialize

# Experiment list builder
display(HTML('<hr style="border: 2px solid #2E86AB; margin: 20px 0;">'))
display(HTML('<h3>🔬 Add Experiments:</h3>'))

exp_name_w = widgets.Text(
    value='Experiment_1',
    description='Name:',
    style=style,
    layout=widgets.Layout(width='400px')
)

exp_input_w = widgets.Text(
    value='C:\\work\\exp1\\images',
    description='Input Dir:',
    style=style,
    layout=widgets.Layout(width='500px')
)

exp_output_w = widgets.Text(
    value='C:\\work\\exp1\\results',
    description='Output Dir:',
    style=style,
    layout=widgets.Layout(width='500px')
)

exp_enabled_w = widgets.Checkbox(
    value=True,
    description='Enabled',
    style={'description_width': 'initial'}
)

exp_custom_w = widgets.Checkbox(
    value=False,
    description='Use custom mapping (override global)',
    style={'description_width': 'initial'}
)

display(exp_name_w, exp_input_w, exp_output_w)
display(widgets.HBox([exp_enabled_w, exp_custom_w]))

add_exp_btn = widgets.Button(
    description='➕ Add Experiment',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

clear_exp_btn = widgets.Button(
    description='🗑️ Clear All',
    button_style='danger',
    layout=widgets.Layout(width='150px')
)

exp_list_out = widgets.Output()

def get_current_mapping():
    """Get current group configuration"""
    groups_map = {}
    colors_map = {}
    
    for i, (name_w, color_w) in enumerate(zip(group_widgets['names'], group_widgets['colors'])):
        group_key = f'Group {i+1}'
        groups_map[group_key] = name_w.value
        colors_map[name_w.value] = color_w.value
    
    # Add special groups
    colors_map['Empty'] = 'black'
    colors_map['Unassigned'] = 'gray'
    
    # Build well_to_group
    well_to_group = {}
    for col, widget in group_widgets['columns'].items():
        assignment = widget.value
        if assignment in groups_map:
            well_to_group[str(col)] = groups_map[assignment]
        else:
            well_to_group[str(col)] = assignment
    
    return well_to_group, colors_map

def add_experiment(b):
    global global_well_mapping_data
    
    with exp_list_out:
        well_to_group, colors = get_current_mapping()
        
        exp_entry = {
            'name': exp_name_w.value,
            'enabled': exp_enabled_w.value,
            'input_dir': exp_input_w.value,
            'output_dir': exp_output_w.value
        }
        
        if exp_custom_w.value or config_mode.value == 'single':
            exp_entry['group_mapping'] = {
                'well_to_group': well_to_group,
                'group_colors': colors
            }
        
        batch_experiments.append(exp_entry)
        
        # Store global mapping
        if not exp_custom_w.value and config_mode.value == 'batch':
            global_well_mapping_data = {
                'well_to_group': well_to_group,
                'group_colors': colors
            }
        
        clear_output(wait=True)
        print(f'\\n✓ Added: {exp_name_w.value}')
        print(f'  Path: {exp_input_w.value}')
        print(f'  Custom: {exp_custom_w.value}')
        print(f'\\n📋 Total experiments: {len(batch_experiments)}\\n')
        
        print('═══════════════════════════════════════════════════')
        for i, exp in enumerate(batch_experiments, 1):
            status = '✓' if exp['enabled'] else '✗'
            custom = ' [CUSTOM]' if 'group_mapping' in exp else ''
            print(f'{i}. {status} {exp["name"]}{custom}')
        print('═══════════════════════════════════════════════════')

def clear_experiments(b):
    global batch_experiments
    batch_experiments = []
    with exp_list_out:
        clear_output(wait=True)
        print('\\n🗑️ Cleared all experiments')

add_exp_btn.on_click(add_experiment)
clear_exp_btn.on_click(clear_experiments)

display(widgets.HBox([add_exp_btn, clear_exp_btn]))
display(exp_list_out)

print('\\n✓ Batch configuration ready. Proceed to export.')


## Step 7: Export Configuration

**Export all parameters to config.json**

In [ ]:
# EXPORT CONFIGURATION SECTION FOR INTERACTIVE_PREVIEW
# Replaces Step 6/7 - handles both single and batch modes

## Step 7: Export Configuration 💾

display(HTML('<h2 style="color: #2E86AB;">💾 Export Configuration</h2>'))
display(HTML('<p><i>Save complete configuration for batch processing</i></p>'))

export_filename_w = widgets.Text(
    value='config.json',
    description='Filename:',
    style=style,
    layout=widgets.Layout(width='400px')
)

display(export_filename_w)

export_btn = widgets.Button(
    description='💾 Export Configuration',
    button_style='success',
    layout=widgets.Layout(width='300px', height='50px')
)

export_output = widgets.Output()

def export_configuration(b):
    with export_output:
        clear_output(wait=True)
        
        print('\\n' + '='*80)
        print('EXPORTING CONFIGURATION')
        print('='*80)
        
        # Build complete config
        if config_mode.value == 'single':
            # SINGLE MODE
            if len(batch_experiments) == 0:
                print('\\n❌ ERROR: Please add at least one experiment!')
                return
            
            exp = batch_experiments[0]  # Use first/only experiment
            
            config = {
                'batch_mode': False,
                'input_dir': exp['input_dir'],
                'output_dir': exp['output_dir']
            }
            
            # Add well mapping
            if 'group_mapping' in exp:
                config['well_to_group'] = exp['group_mapping']['well_to_group']
                config['group_colors'] = exp['group_mapping']['group_colors']
            
            print(f'\\n✓ Mode: SINGLE EXPERIMENT')
            print(f'  Name: {exp["name"]}')
            
        else:
            # BATCH MODE
            if len(batch_experiments) == 0:
                print('\\n❌ ERROR: Please add at least one experiment!')
                return
            
            config = {
                'batch_mode': True,
                'use_global_well_mapping': use_global_mapping.value,
                'experiments': batch_experiments
            }
            
            # Add global mapping if used
            if use_global_mapping.value and global_well_mapping_data:
                config['global_well_mapping'] = global_well_mapping_data
            
            print(f'\\n✓ Mode: BATCH ({len(batch_experiments)} experiments)')
            print(f'  Global mapping: {use_global_mapping.value}')
        
        # Add all other parameters
        sato_sigmas = list(np.linspace(sato_min.value, sato_max.value, sato_steps.value))
        
        config.update({
            'brightness_contrast': {
                'actin': {
                    'brightness': bright_actin.value,
                    'contrast': contrast_actin.value,
                    'gamma': gamma_actin.value
                },
                'nuclei': {
                    'brightness': bright_nuclei.value,
                    'contrast': contrast_nuclei.value,
                    'gamma': gamma_nuclei.value
                }
            },
            'segmentation': {
                'cell': {
                    'diameter': cell_diameter.value,
                    'flow_threshold': cell_flow_thresh.value,
                    'cellprob_threshold': cell_prob_thresh.value
                },
                'nucleus': {
                    'diameter': nucleus_diameter.value,
                    'flow_threshold': nucleus_flow_thresh.value,
                    'cellprob_threshold': nucleus_prob_thresh.value
                }
            },
            'nucleus_filtering': {
                'min_area': min_nucleus_area.value,
                'max_area': max_nucleus_area.value,
                'min_solidity': min_nucleus_solidity.value,
                'min_eccentricity': min_nucleus_eccentricity.value,
                'max_eccentricity': max_nucleus_eccentricity.value,
                'min_roundness': min_nucleus_roundness.value,
                'min_mean_intensity': min_nucleus_intensity.value,
                'max_intensity_std': max_intensity_std.value
            },
            'cell_filtering': {
                'min_area': min_cell_area.value,
                'max_area': max_cell_area.value,
                'min_solidity': min_cell_solidity.value,
                'max_eccentricity': max_cell_eccentricity.value,
                'reject_multiple_nuclei': reject_multiple_nuclei.value,
                'border_buffer': border_buffer.value,
                'require_valid_nucleus': True
            },
            'skeletonization': {
                'gaussian_sigma': gaussian_sigma.value,
                'sato_sigmas': sato_sigmas,
                'binary_threshold_percentile': threshold_pct.value,
                'min_object_size': min_obj_size.value,
                'closing_radius': closing_radius.value,
                'min_branch_length': min_branch_length.value
            },
            'cortical_analysis': {
                'erosion_pixels': 8,
                'use_sato_for_intensity': True,
                'min_cortical_area': 50
            },
            'vieira_preprocessing': {
                'enable': True,
                'sigma': vieira_sigma.value
            },
            'visualization': {
                'generate_cell_panels': False,  # Disable for large datasets
                'generate_polar_maps': False,   # Disable for large datasets
                'generate_summary_plots': True,
                'generate_validation_plots': True,
                'max_cells_to_visualize': 20,  # Limit per-cell viz
                'experiment_level_only': True,  # NEW: Focus on aggregated plots
                'panel_types': ['summary_tables', 'validation_plots', 'experiment_comparison']
            },
            'processing': {
                'save_masks': False,  # Save disk space for large batches
                'save_skeletons': False,
                'save_comparisons': False,
                'use_gpu': True,
                'batch_size': 8
            }
        })
        
        # Save to file
        output_file = Path(export_filename_w.value)
        with open(output_file, 'w') as f:
            json.dump(config, f, indent=2)
        
        print(f'\\n✓ Saved to: {output_file.absolute()}')
        
        # Print summary
        print('\\n' + '='*80)
        print('CONFIGURATION SUMMARY')
        print('='*80)
        
        if config['batch_mode']:
            print(f'\\nMode: BATCH ({len(config["experiments"])} experiments)')
            print(f'Global mapping: {config["use_global_well_mapping"]}')
            print('\\nExperiments:')
            for i, exp in enumerate(config['experiments'], 1):
                status = '✓' if exp['enabled'] else '✗'
                custom = ' [CUSTOM]' if 'group_mapping' in exp else ''
                print(f'  {i}. {status} {exp["name"]}{custom}')
                print(f'      → {exp["input_dir"]}')
        else:
            print(f'\\nMode: SINGLE EXPERIMENT')
            print(f'Input:  {config["input_dir"]}')
            print(f'Output: {config["output_dir"]}')
        
        # Print well mapping
        if config.get('well_to_group'):
            print('\\nWell Mapping:')
            for col in sorted(config['well_to_group'].keys(), key=lambda x: int(x)):
                print(f'  Column {col:>2} → {config["well_to_group"][col]}')
        elif config.get('global_well_mapping'):
            print('\\nGlobal Well Mapping:')
            for col in sorted(config['global_well_mapping']['well_to_group'].keys(), key=lambda x: int(x)):
                print(f'  Column {col:>2} → {config["global_well_mapping"]["well_to_group"][col]}')
        
        print('\\n' + '='*80)
        print('READY TO RUN!')
        print('='*80)
        
        if config['batch_mode']:
            print(f'\\nCommand:')
            print(f'  python main_pipeline_enhanced.py --config {export_filename_w.value}')
        else:
            print(f'\\nCommand:')
            print(f'  python main_pipeline_enhanced.py --config {export_filename_w.value}')
        
        print('\\n✓ Configuration export complete!\\n')

export_btn.on_click(export_configuration)

display(export_btn)
display(export_output)

display(HTML('<hr style="border: 3px solid #2E86AB; margin: 30px 0;">'))
display(HTML('<h3 style="color: green;">✅ Interactive Configuration Complete!</h3>'))
display(HTML('<p>Export your configuration and run main_pipeline_enhanced.py</p>'))
